<span style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">An Exception was encountered at '<a href="#papermill-error-cell">In [7]</a>'.</span>

# 02 — Transformer (ModernBERT, DeBERTa…)

Stessi articoli, stessi fold per gruppo e stesse metriche del classificatore a feature. Input: il testo
dell'articolo, i primi `MAX_TOKEN` token.

**Versione 2.** Nel training entrano anche le testate di solo training. A ogni epoca, per ogni gruppo di
training, si pescano al più `MAX_TESTATE_PER_GRUPPO` testate (sempre diverse da un'epoca all'altra: il **tetto a
rotazione**), e di ognuna `ARTICOLI_PER_TESTATA` articoli. Così un gruppo grande (Natural News) non pesa mai più
di 5 testate in un'epoca, ma nel corso delle epoche il modello le vede tutte. I pesi di classe si calcolano sui
pesi delle testate. Validazione e test solo sulle testate valutate. `SOLO_VALUTAZIONE = True` riproduce la prima
versione. Se la memoria della GPU non basta (il server è condiviso), il fold riparte con il batch dimezzato e lo
stesso batch effettivo (più accumulo del gradiente).

**La ricetta** (uguale per tutti i modelli, fissata prima dei risultati):

- a ogni epoca si pescano `ARTICOLI_PER_TESTATA` articoli di ogni testata di training (sempre diversi): le
  etichette vere sono una per testata, e tanti articoli della stessa testata insegnano a memoria la testata;
- **pesi di classe** nella cross-entropy (inversi della frequenza delle testate di training per livello);
- AdamW, learning rate `LR` con warmup lineare (`WARMUP` dei passi) e poi decadimento lineare, weight decay;
- **validazione** su circa 1/7 dei gruppi di training (stratificata per livello, mai il test); dopo ogni epoca
  la F1 macro per testata (media delle probabilità) sulla validazione; si tiene l'epoca migliore, ci si ferma
  dopo `PATIENCE` epoche senza miglioramenti, al massimo `EPOCHE_MAX`. Con `PATIENCE = 0` niente early stopping:
  si addestra per `EPOCHE_MAX` epoche e si tiene l'ultima (la validazione resta solo come registro);
- precisione mista (fp16), batch `BATCH` con accumulo fino a `BATCH_EFFETTIVO`, padding dinamico.

Si salva **tutto**, per poter ricalcolare qualsiasi metrica senza riaddestrare:

- per ogni articolo del test: `article_id`, `domain`, `fold`, `y_true`, i **logit** `logit_l1..logit_l5`, le
  **probabilità** `p_l1..p_l5`, `y_pred`;
- `metriche.json`: per articolo, per testata col voto e per testata con la media delle probabilità —
  accuratezza, F1 macro e pesata, precisione/recall/F1 per livello, matrice di confusione, tolerant 0.5 e 1,
  errore medio in livelli, binario (soglia $l_2$/$l_3$) con TP, TN, FP, FN, precisione, recall, F1.

In più, per ogni fold: la loss e le metriche di validazione per epoca, e le predizioni di validazione
dell'epoca scelta.

File in `Results/2-classificazione/transformer/<NOME>/`: `config.json`, `fold<k>_test.csv`, `fold<k>_val.csv`,
`fold<k>_epoche.csv`, alla fine `articoli.csv`, `metriche.json` e `FATTO`. Ogni fold finito si salta se il
notebook viene rilanciato. Log: `Results/2-classificazione/avanzamento_<NOME>.log`.

**Sul server** lo lancia `coda.py`; a mano:
`papermill -k johnvenv 02_transformer.ipynb ../Results/2-classificazione/esecuzioni/<NOME>.ipynb -p NOME ... -p MODELLO ...`

In [1]:
import gc
import json
import math
import random
import time

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedGroupKFold
from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup

from comune import CLASSI, RESULTS, carica, metriche, per_testata_media, registratore

/home/fabio.pinelli/John/johnVenv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
NOME = "modernbert_base_1024_s42"
MODELLO = "answerdotai/ModernBERT-base"
MAX_TOKEN = 1024
BATCH = 16                 # esempi per passo sulla GPU
BATCH_EFFETTIVO = 32       # con accumulo del gradiente
BATCH_EVAL = 32
ARTICOLI_PER_TESTATA = 20
MAX_TESTATE_PER_GRUPPO = 5 # per gruppo, a ogni epoca (tetto a rotazione)
SOLO_VALUTAZIONE = False   # True: training solo sulle testate valutate (come la prima versione)
EPOCHE_MAX = 8
PATIENCE = 2               # 0 = epoche fisse (EPOCHE_MAX), senza early stopping
LR = 2e-5
WARMUP = 0.1
WEIGHT_DECAY = 0.01
SEED = 42
SCHEMA = "fold_gruppo"
FOLDS = 0                  # 0 = tutti; k = solo il fold k
FP16 = True
PROVA = False              # True: 40 testate, 2 epoche, risultati in Results/2-classificazione/prova/

In [3]:
# Parameters
NOME = "modernbert_large_1024_ep2_s1"
MODELLO = "answerdotai/ModernBERT-large"
MAX_TOKEN = 1024
BATCH = 2
SEED = 1
PATIENCE = 0
EPOCHE_MAX = 2


In [4]:
def fissa_seed(s):
    s = int(s)                 # numpy int64 non va bene per random.seed
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)


base = RESULTS / ("prova" if PROVA else "")
OUT = base / "transformer" / NOME
OUT.mkdir(parents=True, exist_ok=True)
registra = registratore(base / f"avanzamento_{NOME}.log")
if PROVA:
    EPOCHE_MAX = 2
config = {k: globals()[k] for k in ["NOME", "MODELLO", "MAX_TOKEN", "BATCH", "BATCH_EFFETTIVO", "BATCH_EVAL", "ARTICOLI_PER_TESTATA",
                                    "MAX_TESTATE_PER_GRUPPO", "SOLO_VALUTAZIONE", "EPOCHE_MAX", "PATIENCE", "LR", "WARMUP", "WEIGHT_DECAY", "SEED",
                                    "SCHEMA", "FP16", "PROVA"]}
config.update({"torch": torch.__version__, "transformers": __import__("transformers").__version__,
               "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"})
json.dump(config, open(OUT / "config.json", "w"), indent=1)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
registra(f"=== {NOME}: {MODELLO}, {MAX_TOKEN} token, seed {SEED}, su {config['gpu']} ===")

art = carica(PROVA, SOLO_VALUTAZIONE)
tok = AutoTokenizer.from_pretrained(MODELLO)
t0 = time.time()
ids = tok(art.text.tolist(), truncation=True, max_length=MAX_TOKEN)["input_ids"]
lunghezze = np.array([len(x) for x in ids])
registra(f"{len(art)} articoli tokenizzati in {time.time() - t0:.0f} s; token per articolo: mediana {int(np.median(lunghezze))}, "
         f"al massimo {MAX_TOKEN} nel {100 * (lunghezze >= MAX_TOKEN).mean():.0f}% dei casi")
PAD = tok.pad_token_id

2026-10-20 09:06:02  === modernbert_large_1024_ep2_s1: answerdotai/ModernBERT-large, 1024 token, seed 1, su Tesla V100S-PCIE-32GB ===


/home/fabio.pinelli/John/johnVenv/lib/python3.12/site-packages/torch/cuda/__init__.py:716: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


2026-10-20 09:07:03  75389 articoli tokenizzati in 57 s; token per articolo: mediana 782, al massimo 1024 nel 31% dei casi


## Gli strumenti

In [5]:
def lotto(indici):
    """Padding dinamico: ogni lotto è lungo quanto il suo articolo più lungo."""
    L = max(lunghezze[i] for i in indici)
    x = np.full((len(indici), L), PAD, dtype=np.int64)
    m = np.zeros((len(indici), L), dtype=np.int64)
    for r, i in enumerate(indici):
        x[r, :lunghezze[i]] = ids[i]; m[r, :lunghezze[i]] = 1
    return torch.from_numpy(x).to(device), torch.from_numpy(m).to(device)


@torch.no_grad()
def predici(model, indici):
    """Logit (float32) per gli articoli indicati, nello stesso ordine; lotti per lunghezza per ridurre il padding."""
    model.eval()
    indici = np.asarray(indici)
    ordine = np.argsort(lunghezze[indici])
    out = np.zeros((len(indici), 5), dtype=np.float32)
    for s in range(0, len(indici), BATCH_EVAL):
        pezzo = ordine[s:s + BATCH_EVAL]
        x, m = lotto(indici[pezzo])
        with torch.autocast("cuda", dtype=torch.float16, enabled=FP16):
            out[pezzo] = model(input_ids=x, attention_mask=m).logits.float().cpu().numpy()
    return out


def tabella(indici, logit):
    sotto = art.loc[indici]
    p = torch.softmax(torch.from_numpy(logit), 1).numpy()
    t = pd.DataFrame({"article_id": sotto.article_id.values, "domain": sotto.domain.values, "fold": sotto[SCHEMA].values,
                      "y_true": sotto.label.values})
    for k in range(5):
        t[f"logit_l{k + 1}"] = logit[:, k]
    for k, c in enumerate(CLASSI):
        t[c] = p[:, k]
    t["y_pred"] = p.argmax(1)
    return t


def f1_testate(t):
    tm = per_testata_media(t)
    return f1_score(tm.y_true, tm.y_pred, average="macro", labels=range(5), zero_division=0)

## Un fold

In [6]:
def un_fold(fold):
    fissa_seed(SEED + fold)
    t_fold = time.time()
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    te = art.index[(art[SCHEMA] == fold) & art.valutazione]
    tr_tutti = art[art[SCHEMA] != fold]
    testate_tr = tr_tutti.drop_duplicates("domain").reset_index(drop=True)
    _, iv = next(StratifiedGroupKFold(7, shuffle=True, random_state=SEED).split(testate_tr, testate_tr.label, testate_tr.gruppo))
    in_val = set(testate_tr.domain.iloc[iv])                          # gruppi interi in validazione
    tr = tr_tutti[~tr_tutti.domain.isin(in_val)]
    va = tr_tutti.index[tr_tutti.domain.isin(in_val) & tr_tutti.valutazione]
    articoli_di = {d: g.index.to_numpy() for d, g in tr.groupby("domain")}
    testate_di = {g: sorted(x.domain.unique()) for g, x in tr.groupby("gruppo")}
    t_tr = tr.drop_duplicates("domain")
    conta = t_tr.groupby("label").peso.sum().reindex(range(5), fill_value=0).to_numpy()   # testate pesate: un gruppo vale al più 5
    pesi = torch.tensor(conta.sum() / (5 * np.maximum(conta, 1e-9)), dtype=torch.float32, device=device)

    def campione_epoca(epoca):
        """Al più MAX_TESTATE_PER_GRUPPO testate per gruppo (diverse a ogni epoca), ARTICOLI_PER_TESTATA articoli ciascuna."""
        rng = np.random.default_rng(SEED * 1000 + fold * 100 + epoca)
        pezzi = []
        for g in sorted(testate_di):
            ds = testate_di[g]
            for d in (rng.choice(ds, MAX_TESTATE_PER_GRUPPO, replace=False) if len(ds) > MAX_TESTATE_PER_GRUPPO else ds):
                pezzi.append(rng.choice(articoli_di[d], min(len(articoli_di[d]), ARTICOLI_PER_TESTATA), replace=False))
        c = np.concatenate(pezzi)
        rng.shuffle(c)
        return c

    campioni = {e: campione_epoca(e) for e in range(1, EPOCHE_MAX + 1)}
    registra(f"fold {fold}: {len(articoli_di)} testate di training ({t_tr.valutazione.sum()} valutate) in {len(testate_di)} gruppi, "
             f"{len(in_val)} di validazione, {art.loc[te].domain.nunique()} di test; {len(campioni[1])} articoli per epoca; "
             f"pesi di classe {np.round(pesi.cpu().numpy(), 2).tolist()}; batch {BATCH}")

    kw = {"reference_compile": False} if "ModernBERT" in MODELLO else {}
    model = AutoModelForSequenceClassification.from_pretrained(MODELLO, num_labels=5, **kw).to(device)
    nodecay = ["bias", "LayerNorm.weight", "norm.weight"]
    gruppi = [{"params": [p for n, p in model.named_parameters() if not any(x in n for x in nodecay)], "weight_decay": WEIGHT_DECAY},
              {"params": [p for n, p in model.named_parameters() if any(x in n for x in nodecay)], "weight_decay": 0.0}]
    opt = torch.optim.AdamW(gruppi, lr=LR)
    accumulo = max(1, BATCH_EFFETTIVO // BATCH)
    passi_totali = sum(math.ceil(len(c) / (BATCH * accumulo)) for c in campioni.values())
    sched = get_linear_schedule_with_warmup(opt, int(WARMUP * passi_totali), passi_totali)
    scaler = torch.amp.GradScaler("cuda", enabled=FP16)
    loss_fn = torch.nn.CrossEntropyLoss(weight=pesi)
    migliore, stato_migliore, logit_val_migliori, attese, righe = (-1.0, np.inf), None, None, 0, []

    for epoca in range(1, EPOCHE_MAX + 1):
        t0 = time.time()
        campione = campioni[epoca]
        model.train(); tot, n = 0.0, 0
        opt.zero_grad()
        for s in range(0, len(campione), BATCH):
            pezzo = campione[s:s + BATCH]
            x, m = lotto(pezzo)
            y = torch.tensor(art.label.values[pezzo], device=device)
            with torch.autocast("cuda", dtype=torch.float16, enabled=FP16):
                loss = loss_fn(model(input_ids=x, attention_mask=m).logits.float(), y)
            scaler.scale(loss / accumulo).backward()
            tot += loss.item() * len(pezzo); n += len(pezzo)
            if (s // BATCH + 1) % accumulo == 0 or s + BATCH >= len(campione):
                scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(opt); scaler.update(); opt.zero_grad(); sched.step()
        logit_val = predici(model, va)
        t_val = tabella(va, logit_val)
        loss_val = float(torch.nn.functional.cross_entropy(torch.from_numpy(logit_val), torch.tensor(art.label.values[va])).item())
        f1_val = f1_testate(t_val)
        acc_val = float((t_val.y_true == t_val.y_pred).mean())
        righe.append({"fold": fold, "epoca": epoca, "loss_train": tot / n, "loss_val": loss_val, "acc_val_articoli": acc_val,
                      "f1_val_testate": f1_val, "esempi": n, "secondi": time.time() - t0})
        registra(f"fold {fold} epoca {epoca}/{EPOCHE_MAX}: loss {tot / n:.4f}, val loss {loss_val:.4f}, val F1 testate {f1_val:.3f}, "
                 f"val acc articoli {acc_val:.3f} ({time.time() - t0:.0f} s)")
        if PATIENCE == 0:                    # epoche fisse: niente early stopping, si tiene l'ultima epoca
            logit_val_migliori = logit_val
            continue
        if (f1_val, -loss_val) > (migliore[0], -migliore[1]):
            migliore, attese = (f1_val, loss_val), 0
            stato_migliore = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            logit_val_migliori = logit_val
        else:
            attese += 1
            if attese >= PATIENCE:
                registra(f"fold {fold}: stop all'epoca {epoca}, la migliore è la {righe[-1 - attese]['epoca']}")
                break
    if stato_migliore is not None:
        model.load_state_dict(stato_migliore)
    t_test = tabella(te, predici(model, te))
    pd.DataFrame(righe).to_csv(OUT / f"fold{fold}_epoche.csv", index=False)
    tabella(va, logit_val_migliori).to_csv(OUT / f"fold{fold}_val.csv", index=False)
    t_test.to_csv(OUT / f"fold{fold}_test.csv", index=False)       # per ultimo: la sua presenza vuol dire "fold finito"
    registra(f"fold {fold}: finito in {(time.time() - t_fold) / 60:.0f} min, test acc articoli {(t_test.y_true == t_test.y_pred).mean():.3f}, "
             f"F1 testate {f1_testate(t_test):.3f}, memoria GPU massima {torch.cuda.max_memory_allocated() / 2**30 if torch.cuda.is_available() else 0:.1f} GB")
    del model, opt, sched, scaler, stato_migliore
    gc.collect(); torch.cuda.empty_cache()

## Tutti i fold

<span id="papermill-error-cell" style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">Execution using papermill encountered an exception here and stopped:</span>

In [7]:
folds = [int(FOLDS)] if FOLDS else [int(f) for f in sorted(art[SCHEMA].unique())]
for fold in folds:
    if (OUT / f"fold{fold}_test.csv").exists():
        registra(f"fold {fold}: già fatto, salto")
        continue
    while True:
        memoria_finita = False
        try:
            un_fold(fold)
        except torch.cuda.OutOfMemoryError:
            memoria_finita = True                  # si libera la memoria fuori dall'except, quando il traceback non c'è più
        if not memoria_finita:
            break
        gc.collect(); torch.cuda.empty_cache()
        if BATCH == 1:
            raise RuntimeError("memoria GPU insufficiente anche con batch 1")
        BATCH, BATCH_EVAL = max(1, BATCH // 2), max(1, BATCH_EVAL // 2)
        registra(f"fold {fold}: memoria GPU insufficiente (server condiviso), riparto con batch {BATCH} "
                 f"(stesso batch effettivo {BATCH_EFFETTIVO}) e batch di valutazione {BATCH_EVAL}")
pred = pd.concat([pd.read_csv(OUT / f"fold{f}_test.csv") for f in sorted(art[SCHEMA].unique()) if (OUT / f"fold{f}_test.csv").exists()])
if len(pred) == int(art.valutazione.sum()):
    pred.to_csv(OUT / "articoli.csv", index=False)
    m = metriche(pred)
    json.dump(m, open(OUT / "metriche.json", "w"), indent=1)
    (OUT / "FATTO").write_text(time.strftime("%Y-%m-%d %H:%M:%S"))
    registra(f"=== {NOME} FINITO: testata (media) F1 {m['testata_media']['f1_macro']:.3f}, acc {m['testata_media']['accuratezza']:.3f}, "
             f"bin {m['testata_media']['bin_accuratezza']:.3f} ===")
    display(pd.DataFrame({k: {x: v[x] for x in ["accuratezza", "f1_macro", "bin_accuratezza", "tolerant_0.5", "tolerant_1"]} for k, v in m.items()}).T.round(3))

2026-10-20 09:07:04  fold 1: 336 testate di training (218 valutate) in 192 gruppi, 114 di validazione, 61 di test; 4727 articoli per epoca; pesi di classe [0.8399999737739563, 1.0099999904632568, 1.809999942779541, 0.5899999737739563, 1.7400000095367432]; batch 2


Some weights of ModernBertForSequenceClassification were not initialized from the model checkpoint at answerdotai/ModernBERT-large and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


/home/fabio.pinelli/John/johnVenv/lib/python3.12/site-packages/torch/cuda/__init__.py:716: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


RuntimeError: NVML_SUCCESS == DriverAPI::get()->nvmlInit_v2_() INTERNAL ASSERT FAILED at "../c10/cuda/CUDACachingAllocator.cpp":963, please report a bug to PyTorch. 